# GridWorld Rules
* Dots are valid cells, number signs are walls
* Your score starts at 0 points. A movement costs -1 points, reaching the goal is 20 points
* Can move up, down, left, right but cannot move through walls and cannot leave the grid
* Your goal is maximimize your score.

# Introduction to Reinforcement Learning (2-18-26)

## Introduction
* An agent learns how to make decisions by trial and error, using rewards and penalties from the envirnoment
* The agent takes an action, the environment responds, The agent gets feedback in the form of a reward (good outcome) or a penalty (bad outcome). Over time the agent learns which actions lead to better long-term outcomes.

## Components

### Agent
* The decision maker
* Examples: robot, game-playing AI, software system
* At each step the agent will:
    * Observe information about the environment 
    * Select an action
    * Recieve a reward and a new observation
    * Update its internal knowledge to improve future decisions
* Note the agent cannot control the environment, directly choose its rewards, or see the future

### Environment
* Everything the agent interacts with.
* Defines the world the agent lives in, the rules of interaction, and the feedback the agent recieves.
* Examples: game, physical world, simulation

### Action
* The choices the agent can make

### Rewards
* Numerical feedback from the environment that tells the agent how good an outcome was.
* Rewards can be positive if the outcome is favorable or negative if the outcome was not favorable (called a penalty)

## Policy
* This is what the agent is learning: "In this situation, take this action."
* Policy improves over time as the agent explores new actions and exploits known actions that have worked before.
* Note that there needs to be a balance of exploration and exploitation to find the best strategy.

## Difference from Supervised Learning
* In supervised learning an algorithm learns by considering examples (labeled data). In reinforcement learning an agent interacts with an environment and learns by taking actions.

## Types of Reinforcement Learning

### Value-Based Methods
* These algorithms learn a value function (how good an action was or the new state of the system is) and derive a policy from it.
* Examples: Q-Learning, SARSA (State-Action-Reward-State-Action), Deep Q-Network

### Policy-Based Methods
* The algorithms learn a policy withoout using a value table.
* Optimize the expected reward via gradient ascent (an optimizer) making them well suited for continuouos spaces
* Often used in robotics and control

### Model-Based Methods
* Algorithms learn or use a model of the environment.
* Agent predicts how an action affect the future state which enables planning.
* More complex and sensitive to modeling errors

## Example Algorithm: Q-Learning
* One of the most widely used algorithms in reinforcement learning. 
* Teaches an agent which actions are best in each state by learning from experience, without needed to model the environment

### Introduction
* Q-learning is model-free, value-based, and off-policy
    * Model-free: The agent does not know the environment's transition or reward rules in advance
    * Value-based: it learns numerical values for actions
    * Off-policy: It learns the optimal policy even while following a different (exploratory) policy

### Q-Value
* Q-value = quality value (Q(s,a))
* It represents the expected future cumulative reward for taking action a in state s, and then acting optimally afterward.
* The algorithm stores these in a Q-table where the rows are the states and the columns are the actions


### How it Works
* The agent starts in a state s
* The agent chooses an action a
* The environment returns:
    * A reward r
    * The next state $s^\prime$
* The agent updates its estimate of Q(s,a)
* Repeat over many episodes

### Temporal Difference Update
$$Q(s,a) \leftarrow Q(s,a) + \alpha[r+\gamma max_{a^\prime Q(s^\prime, a^\prime) - Q(s,a)}]$$
* s: current state
* a: action taken
* r: reward recieved:
* $s^\prime$: next state
* $\alpha$: Learning rate
* $\gamma$: discount factor (how much to value future rewards)

### Exploration vs. Exploitation 
* Q-Learning needs to balance exploration (trying other actions to discover better strategies) and explotation (choosing the action with the highest Q-value)
    * Too much exploration can lead to poor short-term performance but too much exploitation can lead to missing better strategies
* Q-Learning Approach: $\epsilon$-greedy:
    * Choose the best known action: probability 1-$\epsilon$
    * Choose a random action: $\epsilon$
* A common approach is to use $\epsilon$ decay where the value of epsilon will decay per step to some threshold.

## Q-Learning in Python

### GridWorld
* A gridworld is a two-dimensional grid of cells where:
    * Each cell represents a state
    * An agent occupies one cell at a time
    * The agent can move between cells using a small set of actions
* Basically a maze

In [ ]:
#############
## IMPORTS ##
#############
import numpy as np
import random
from collections import defaultdict

In [ ]:
###########################
## GRIDWORLD ENVIRONMENT ##
###########################
class Gridworld:
    """
    Simple deterministic gridworld.
    - 'S' start
    - 'G' goal
    - '#' walls/obstacles
    - step reward: -1 per move (encourages short paths)
    - goal reward: +10 and terminal
    """
    def __init__(self, grid, start, goal, step_reward=-1.0, goal_reward=10.0):
        """
        Inputs:
            grid: list of strings representing the grid layout
            start: (row, col) tuple for start position
            goal: (row, col) tuple for goal position
            step_reward: reward for each step taken
            goal_reward: reward for reaching the goal
        Returns:
            None
        Initializes the gridworld environment with the given parameters.
        """
        # Store the grid and parameters
        self.grid = grid
        # Compute dimensions
        self.H = len(grid)
        self.W = len(grid[0])
        self.start = start
        self.goal = goal
        self.step_reward = step_reward
        self.goal_reward = goal_reward

        # Define action space: 0=up, 1=down, 2=left, 3=right
        self.actions = {
            0: (-1, 0),  # up
            1: (1, 0),   # down
            2: (0, -1),  # left
            3: (0, 1),   # right
        }
        # Initialize state
        self.reset()

    def reset(self):
        """
        Inputs:
            None
        Returns:            
            state: initial state (row, col) tuple
        Resets the environment to the initial state and returns it.
        """
        self.state = self.start
        return self.state

    def in_bounds(self, r, c):
        """
        Inputs:
            r: row index
            c: column index
        Returns:
            True if (r, c) is within the grid bounds, False otherwise
        Determines if the given position is within the grid boundaries.
        """
        return 0 <= r < self.H and 0 <= c < self.W

    def passable(self, r, c):
        """
        Inputs:
            r: row index
            c: column index
        Returns:
            True if (r, c) is not a wall, False otherwise
        Determines if the given position is passable (not a wall).
        """
        # Check if the cell is not a wall. If the cell (r,c) is a "#" then it's a wall and not passable.
        return self.grid[r][c] != "#"

    def step(self, action):
        """
        Inputs:
            action: integer in {0, 1, 2, 3} representing the action to take
        Returns:
            next_state: (row, col) tuple of the new state after taking the action
            reward: float reward received after taking the action
            done: boolean indicating if the episode has ended (reached goal)
        Executes the given action, updates the state, and returns the new state, reward, and done flag.
        """
        # Compute the change in position based on the action
        dr, dc = self.actions[action]
        # Get the current position
        r, c = self.state
        # Compute the new position
        nr = r + dr
        nc = c + dc

        # If invalid move, stay in place. If the new position is out of bounds or not passable, 
        # we do not move and stay in the current position.
        if not self.in_bounds(nr, nc) or not self.passable(nr, nc):
            nr = r
            nc = c

        # Update the state to the new position
        self.state = (nr, nc)

        # Terminal check. If the new state is the goal, return the goal reward and done=True. 
        # Otherwise, return the step reward and done=False.
        if self.state == self.goal:
            return self.state, self.goal_reward, True

        return self.state, self.step_reward, False

In [ ]:
######################
## Q-LEARNING AGENT ##
######################

## Epsilon-greedy action selection
def epsilon_greedy(Q, state, n_actions, epsilon):
    """
    Inputs:
        Q: dict mapping states to arrays of action values
        state: current state
        n_actions: number of possible actions
        epsilon: exploration rate
    Returns:
        action: integer action chosen by epsilon-greedy policy
    Chooses an action using epsilon-greedy policy based on the Q-values for the given state.
    """
    # With probability epsilon, choose a random action. Otherwise, choose the action with 
    # the highest Q-value.
    if random.random() < epsilon:
        return random.randrange(n_actions)
    # Otherwise, choose the action with the highest Q-value for the current state. If there are 
    # ties, randomly select among the best actions.
    qs = Q[state]
    max_q = np.max(qs)
    best = np.where(qs == max_q)[0]
    return int(np.random.choice(best))

## Q-learning algorithm
def q_learning(env, episodes=2000, alpha=0.1, gamma=0.99,
               epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995,
               max_steps=500, seed=67):
    """
    Inputs:
        env: Gridworld environment instance
        episodes: number of episodes to train
        alpha: learning rate
        gamma: discount factor
        epsilon_start: initial exploration rate
        epsilon_end: minimum exploration rate
        epsilon_decay: multiplicative decay factor for epsilon per episode
        max_steps: maximum steps per episode to prevent infinite loops
        seed: random seed for reproducibility
    Returns:
        Q: learned Q-values as a dict mapping states to action value arrays
        returns: array of total rewards per episode
    Trains a Q-learning agent on the given environment and returns the learned Q-values and episode returns.
    """
    # Set random seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)

    # Number of actions in the environment (up, down, left, right)
    n_actions = 4

    # defaultdict gives unseen states a zero-vector of action values. Basically this line 
    # initializes the Q-table as a dictionary where each key is a state and the value is an 
    # array of action values initialized to zero.
    Q = defaultdict(lambda: np.zeros(n_actions, dtype=float))

    # Initialize epsilon for exploration
    epsilon = epsilon_start
    returns = []   # total reward per episode

    # Main training loop
    for ep in range(episodes):
        # Reset the environment at the start of each episode and initialize total reward. 
        # We call env.reset() to start a new episode and get the initial state. We also initialize 
        # total_reward to 0 for this episode.
        s = env.reset()
        total_reward = 0.0
        # Step through the episode until done or max_steps reached. We use a for loop to iterate up to max_steps. 
        # In each step, we select an action using the epsilon-greedy policy, take a step in the environment, 
        # and update the Q-values based on the observed reward and next state.
        for t in range(max_steps):
            # Select action using epsilon-greedy policy. We call the epsilon_greedy function to select an action 
            # based on the current Q-values and exploration rate.
            a = epsilon_greedy(Q, s, n_actions, epsilon)
            # Take action and observe next state, reward, and done flag. We call env.step(a) to execute the 
            # action in the environment and receive the next state, reward, and done flag.
            s2, r, done = env.step(a)
            total_reward += r

            # Q-learning update (off-policy). We compute the TD target as the observed reward plus the discounted
            # maximum Q-value of the next state (if not done). TD means temporal difference, and the TD target is 
            # the estimate of the return based on the observed reward and the estimated value of the next state.
            td_target = r + (0.0 if done else gamma * np.max(Q[s2]))
            td_error = td_target - Q[s][a]
            Q[s][a] += alpha * td_error

            # Move to the next state. We update our current state to the next state for the next iteration of the loop.
            s = s2
            if done:
                break
        # After the episode ends, we append the total reward received in this episode to the returns list. 
        # This allows us to track the performance of the agent over time.
        returns.append(total_reward)

        # decay epsilon. We decay the exploration rate epsilon after each episode by multiplying it with epsilon_decay, 
        # but we also ensure it does not go below epsilon_end.
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, np.array(returns)

In [ ]:
##############################
## GREEDY POLICY EXTRACTION ##
##############################
def greedy_policy(Q, env):
    """
    Inputs:
        Q: dict mapping states to arrays of action values
        env: Gridworld environment instance (used to determine valid states)
    Returns:
        policy: dict mapping states to the greedy action (integer) based on Q-values
    Extracts a greedy policy from the learned Q-values. For each valid state in the environment, 
    it selects the action with the highest Q-value. The policy is returned as a dictionary mapping states
    to actions.
    """
    # We iterate through all the cells in the grid. If the cell is not a wall, we consider it a valid state.
    policy = {}
    # For each valid state, we look up the Q-values for that state and select the action with the highest Q-value.
    for r in range(env.H):
        for c in range(env.W):
            # If the cell is a wall, we skip it since it's not a valid state for the agent to be in.
            if env.grid[r][c] == "#":
                continue
            # For valid states, we look up the Q-values and select the action with the highest Q-value. 
            # We store this in the policy dictionary.
            s = (r, c)
            # We use np.argmax to find the index of the action with the highest Q-value for state s. This 
            # gives us the greedy action for that state.
            a = int(np.argmax(Q[s]))
            # We store the greedy action for state s in the policy dictionary.
            policy[s] = a
    return policy

In [ ]:
##################
## ROLLOUT PATH ##
##################
def rollout_path(env, policy, max_steps=200):
    """
    Inputs:
        env: Gridworld environment instance
        policy: dictionary mapping states to actions
        max_steps: maximum number of steps to take in the rollout
    Returns:
        path: list of (row, col) tuples representing the path taken by the agent
    Executes a rollout following the given policy and returns the path taken by the agent.
    """
    # Start from the initial state and follow the policy until we reach the goal or exceed max_steps.
    s = env.reset()
    # We initialize the path with the starting state. Then, for each step, we look up the action from 
    # the policy for the current state, execute it, and add the resulting state to the path. We also check if
    #  the episode has ended (done) after each step, and if so, we break out of the loop.
    path = [s]
    for _ in range(max_steps):
        a = policy[s]
        s2, _, done = env.step(a)
        path.append(s2)
        s = s2
        if done:
            break
    return path

In [ ]:
###################
## VISUALIZATION ##
###################

## For visualization of the policy, we can define a mapping from action indices to arrow symbols. 
# This will allow us to print the policy in a more intuitive way, showing the direction of the action.
ARROWS = {0: "↑", 1: "↓", 2: "←", 3: "→"}

## Grid visualization with policy
def print_policy(env, policy):
    """
    Inputs:
        env: Gridworld environment instance
        policy: dictionary mapping states to actions
    Returns:
        None
    Prints the grid with the greedy policy actions represented as arrows. The start and goal positions 
    are marked with "S" and "G" respectively, and walls are marked with "#".
    """
    # We iterate through each cell in the grid and check if it's the start, goal, or a wall. If it's the start 
    # or goal, we mark it with "S" or "G". If it's a wall, we mark it with "#". Otherwise, we look up the action 
    # from the policy for that state and represent it with an arrow. Finally, we print the grid with the policy 
    # actions.
    out = []
    for r in range(env.H):
        row = []
        for c in range(env.W):
            cell = env.grid[r][c]
            if (r, c) == env.start:
                row.append("S")
            elif (r, c) == env.goal:
                row.append("G")
            elif cell == "#":
                row.append("#")
            else:
                row.append(ARROWS[policy[(r, c)]])
        out.append(" ".join(row))
    print("\nGreedy policy:")
    print("\n".join(out))

## Grid visualization with path
def print_path(env, path):
    """
    Inputs:
        env: Gridworld environment instance
        path: list of (row, col) tuples representing the path taken by the agent
    Returns:
        None
    Prints the grid with the path taken by the agent marked with "*". The start and goal 
    positions are marked with "S" and "G" respectively, and walls are marked with "#".
    """
    # Create a copy of the grid characters to modify for visualization. This will allow us to mark the 
    # path without altering the original grid.
    grid_chars = [row[:] for row in env.grid]
    # Mark the path on the grid. We iterate through the path and mark each cell with "*" except for the 
    # start and goal positions. We also check if the cell is not a wall before marking it.
    for (r, c) in path:
        # We only mark the cell if it's not the start or goal and it's not a wall. This way we can visualize 
        # the path taken by the agent without overwriting important information about the grid.
        if (r, c) != env.start and (r, c) != env.goal and grid_chars[r][c] != "#":
            grid_chars[r][c] = "*"

    # Print the grid with the path. We iterate through each cell and print "S" for the start, "G" for the goal, 
    # "#" for walls, and "*" for the path taken by the agent. This will give us a visual representation of the 
    # path on the grid.
    out = []
    for r in range(env.H):
        row = []
        for c in range(env.W):
            if (r, c) == env.start:
                row.append("S")
            elif (r, c) == env.goal:
                row.append("G")
            else:
                row.append(grid_chars[r][c])
        out.append(" ".join(row))
    # Finally, we print the grid with the path. The output will show the grid layout with the path taken by the 
    # agent marked with "*", making it easy to visualize how the agent navigated from the start to the goal.
    print("\nGreedy rollout path (* marks visited cells):")
    print("\n".join(out))

## Learning curve visualization
def print_learning_curve(returns, window=50):
    """
    Inputs:
        returns: array of total rewards per episode
        window: integer size of the moving average window
    Returns:
        None
    Prints a summary of the learning curve using a moving average of the returns for readability.
    """
    # moving average for readability. This computes a moving average of the returns using a convolution 
    # with a window of ones. The mode "valid" means that it only computes the average for positions where 
    # the full window fits, which effectively smooths the learning curve. The moving average helps to visualize 
    # the overall trend in the returns over episodes, especially when there is a lot of variability. 
    if len(returns) < window:
        print("Not enough episodes for moving average.")
        return
    # compute the moving average using a convolution with a window of ones normalized by the window size. 
    # This will give us a smoothed version of the returns that we can use to summarize the learning curve.
    ma = np.convolve(returns, np.ones(window)/window, mode="valid")
    print(f"\nLearning curve (moving average over {window} episodes):")
    print(f"  start: {ma[0]:.2f}  mid: {ma[len(ma)//2]:.2f}  end: {ma[-1]:.2f}")


In [30]:
###################
## EXAMPLE USAGE ##
###################
# Define a grid (list of lists of characters). Note the inner lists are mutable, 
# which allows us to modify the grid when printing paths. This does mean we have to 
# use list() to convert the strings into lists of characters.
# S = start, G = goal, # = wall, . = empty cell
grid = [
    list("...#...."),
    list("...#..#."),
    list("...#..#."),
    list("...#...."),
    list(".....#.."),
    list("##..##.."),
    list("....#..."),
]
# Some other example grids you can try:
#grid = [
#    list("..#.."),
#    list("..#.."),
#    list("....."),
#    list(".###."),
#    list("..#.."),
#]

#grid = [
#    list("...#......"),
#    list("...#..##.."),
#    list("...#......"),
#    list("######...."),
#    list("......#.."),
#    list("..##..#..#"),
#    list("......#.."),
#    list("..####...."),
#]

#grid= [
#    list("....#......."),
#    list("....#..###.."),
#    list("....#......."),
#    list("######..#..."),
#    list("......#.#.."),
#    list("..##..#.#..#"),
#    list("......#...."),
#    list("..#######..."),
#    list("......#...."),
#    list("...#........"),
#]

# Define start and goal positions as (row, col) tuples. These can be changed to test different scenarios.
start = (0, 0)
goal  = (6, 7)

# Create the Gridworld environment with the specified grid, start, and goal. You can also adjust the 
# step_reward and goal_reward if desired. The step_reward encourages the agent to find shorter paths, 
# while the goal_reward provides a strong incentive to reach the goal. 
env = Gridworld(grid, start, goal, step_reward=-1.0, goal_reward=20.0)

# Train the Q-learning agent on the environment and obtain the learned Q-values and episode returns. 
# You can adjust the learning rate (alpha), discount factor (gamma), exploration parameters 
# (epsilon_start, epsilon_end, epsilon_decay), and the number of episodes and max steps to see how 
# it affects learning.
Q, returns = q_learning(
    env,
    episodes=400,
    alpha=0.15,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    max_steps=30,
    seed=42
)

# Print the learning curve to see how the total rewards per episode evolved during training. The 
# moving average helps to visualize trends in the learning process.
print_learning_curve(returns, window=100)

# Extract the greedy policy from the learned Q-values and print it. The policy will show the best action 
# to take from each state according to the learned Q-values. The arrows indicate the direction of the action 
# (up, down, left, right), while walls and the start/goal positions are represented by their respective symbols.
policy = greedy_policy(Q, env)
print_policy(env, policy)

# Perform a rollout using the greedy policy to see the path taken from the start to the goal. The path will be 
# marked on the grid, and you can check if the agent successfully reached the goal and how many steps it took.
path = rollout_path(env, policy, max_steps=200)
print_path(env, path)

# Finally, print the length of the path and whether the goal was reached to evaluate the performance of the 
# learned policy.
print(f"\nPath length: {len(path)-1} steps")
print(f"Reached goal? {'Yes' if path[-1] == goal else 'No'}")


Learning curve (moving average over 100 episodes):
  start: -30.00  mid: -20.21  end: 2.30

Greedy policy:
S ↓ ↓ # ↑ → ↓ ←
↓ ↓ ↓ # → ↓ # ↓
↓ ↓ ↓ # ↓ ↓ # ↓
→ → ↓ # → → → ↓
→ → → → ↑ # → ↓
# # ↑ ↑ # # → ↓
↑ ← ↓ ↑ # ↑ → G

Greedy rollout path (* marks visited cells):
S . . # . . . .
* . . # . . # .
* . . # . . # .
* * * # * * * *
. . * * * # . *
# # . . # # . *
. . . . # . . G

Path length: 15 steps
Reached goal? Yes
